# CardioSense — Pipeline 2b: ECG Signal + PTB-XL+ Feature Fusion (v2)

**Goal:** beat the base paper on PTB-XL diagnostic superclasses (multi-label,
standard folds 1–8 / 9 / 10):

| Paper model | Macro F1 | Macro AUC |
|---|---|---|
| features-only (Uni-G + 12SL, dense net) | 0.73 | 0.92 |
| signal-only CNN | 0.75 | 0.93 |
| combined | "best, marginal gain" | — |

The paper's conclusion is that *"integrating raw signals requires more
sophisticated architectures to yield significant benefits"*. This notebook
does exactly that:

| Exp | Model | What changed vs the paper |
|---|---|---|
| E-D1 | Feature MLP (BN + SiLU, 3-seed avg) | stronger dense net, outlier clipping, + demographics |
| E-D2 | **XGBoost** on the same features | gradient-boosted trees are the strongest tabular learner; handles NaN natively |
| E-D3 | Feature ensemble (MLP + XGB) | — |
| E-S | **SE-ResNet-1D** signal branch (aux head of E-J) | residual + squeeze-excitation, OneCycle, 3-seed avg |
| E-J | **Gated joint network** (signal + features, end-to-end) | learned per-dimension gate, modality dropout, deep supervision |
| E-E | AUC-weighted log-odds fusion | kept from v1 |
| E-M | Mean-logit ensemble | — |
| E-K | **Stacked meta-learner** (per-class LogReg on all branches) | fit on val with 5-fold OOF; test untouched |

The **final model is chosen by validation** macro AUC, never by test.

**No need to run `00_colab_setup.ipynb` first**: §1 sets itself up in about a minute
(only PTB-XL must already be on Drive). `02_ecg_training.ipynb`'s CNN is
*optional* (used as one more stacking input if its checkpoint is found).
**Needs a GPU** (T4 is enough; ~25–35 min end to end with 3 seeds).

## 1. Environment Setup

In [ ]:
# Fast, self-contained setup: this notebook does NOT need 00_colab_setup.ipynb.
# It mounts Drive, clones the code, and installs only the two packages Colab lacks.
# (00_colab_setup also downloads ~45 GB of chest X-rays and asks for kaggle.json,
# which is what makes it so slow; none of that is needed here.)
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_DIR = Path("/content/CardioSense")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/artrai73/CardioSense.git", str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    os.environ["CARDIOSENSE_DATA_ROOT"] = "/content/drive/MyDrive/CardioSense/data"

    # numpy, pandas, scipy, scikit-learn, xgboost, torch and pyyaml ship with Colab
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wfdb>=4.1", "captum>=0.7"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
else:
    here = Path.cwd()
    while not (here / "pyproject.toml").exists() and here != here.parent:
        here = here.parent
    os.chdir(here)

SRC = Path.cwd() / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

data_root = Path(os.environ.get("CARDIOSENSE_DATA_ROOT", "data"))
ptbxl_csv = data_root / "ecg" / "ptbxl" / "ptbxl_database.csv"
print("Working directory:", Path.cwd())
print("Data root        :", data_root)
print("PTB-XL on Drive  :", "found" if ptbxl_csv.exists() else
      "MISSING - run only section 7b of 00_colab_setup.ipynb once")

### Run settings — the only cell to edit for the ablation runs

| Run | `USE_DEMOGRAPHICS` | `N_SEEDS` | Time on a T4 | Answers |
|---|---|---|---|---|
| Main result (done) | `True` | `3` | ~15 min | 0.772 F1 / 0.942 AUC |
| **Ablation A** | `False` | `3` | ~15 min | Do we still beat the paper with *only* its features (no age/sex)? |
| **Ablation B** | `True` | `1` | ~3 min | How much comes from averaging 3 seeds vs. one model? |

Ablation B reuses the seed-42 joint network already saved on Drive, so it does not retrain it.
Each run writes to its own folder: `MyDrive/CardioSense/results/ecg_fusion/<run name>/`.

In [ ]:
USE_DEMOGRAPHICS = True   # False = the paper's pure Uni-G + 12SL feature set (Ablation A)
N_SEEDS = 3                # 1 = single model (Ablation B), 3 = main result

RUN_TAG = f"{'with' if USE_DEMOGRAPHICS else 'no'}_demographics__{N_SEEDS}seed{'s' if N_SEEDS > 1 else ''}"
print("run:", RUN_TAG)

## 2. Imports

In [ ]:
import sys
from pathlib import Path

SRC = Path.cwd() / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import cardiosense
print("CardioSense imported from:", cardiosense.__file__)

In [ ]:
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from IPython.display import display

from cardiosense.common.config import load_config
from cardiosense.common.env import get_device, print_environment
from cardiosense.common.io_utils import save_json
from cardiosense.common.paths import PATHS
from cardiosense.common.seeding import set_seed
from cardiosense.common.training import count_parameters

from cardiosense.ecg.data import (
    build_superclass_labels, load_metadata, resolve_ptbxl_root, split_by_fold, verify_dataset,
)
from cardiosense.ecg.preprocessing import build_waveform_cache
from cardiosense.ecg.dataset import build_dataloaders, compute_pos_weight
from cardiosense.ecg.models import build_model
from cardiosense.ecg import evaluate as ev

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 40)
_ = print_environment()

## 3. Configuration

Same `ecg` config as `02_ecg_training.ipynb` — same classes, same folds, same
saved-model paths, so the signal branch loaded below is guaranteed to match.

In [ ]:
cfg = load_config("ecg")

SEED = int(cfg.seed)
set_seed(SEED, strict=bool(cfg.get("strict_determinism", False)))
device = get_device()

RESULTS = PATHS.root / cfg.output.results_dir
MODELS = PATHS.root / cfg.output.models_dir
# one results folder per run (on Drive in Colab), so ablation runs never overwrite each other
FUSION_RESULTS = (Path("/content/drive/MyDrive/CardioSense/results/ecg_fusion") if IN_COLAB
                  else RESULTS.parent / "ecg_fusion") / RUN_TAG
FUSION_RESULTS.mkdir(parents=True, exist_ok=True)

CLASSES = list(cfg.task.classes)
use_amp = bool(cfg.training.get("amp", True)) and device.type == "cuda"

print("classes:", CLASSES)
print("device :", device)
print("results:", RESULTS)
print("models :", MODELS)

## 4. Restore ECG state (dataset, labels, splits, waveforms)

Same restore pattern as `02_ecg_training.ipynb` §4–7 — rebuilds `database`,
`labels`, `splits`, `waveforms` from the cached files rather than re-parsing
the raw WFDB records. `splits[name]["database"]` (indexed by `ecg_id`) is the
join key for the PTB-XL+ features below.

In [ ]:
root = resolve_ptbxl_root(cfg)
print(json.dumps(verify_dataset(cfg, root), indent=2))

database, statements = load_metadata(cfg, root)
database, labels, label_report = build_superclass_labels(database, statements, cfg)

waveforms, cache_path = build_waveform_cache(database, cfg, root=root)
print(f"waveforms: {waveforms.shape} ({waveforms.dtype})")

splits = split_by_fold(database, labels, cfg)
print("patient overlap between splits (must be all zero):", splits["patient_overlap"])
for name in ("train", "val", "test"):
    print(f"  {name}: {splits[name]['summary']['n_records']} records")

## 5. Load the trained signal-only CNN

Loads the checkpoint saved by `02_ecg_training.ipynb` §16 — this is the
`pos_weight` + oversampling CNN (macro F1 0.7146, macro AUC 0.9087 on the
full test set, from the last training run). Not retrained here.

The sanity-check print below should roughly match that number. If it doesn't,
`02_ecg_training.ipynb` §16 (Save Model) needs to be re-run after your most
recent training run, or this notebook is loading a stale checkpoint.

In [ ]:
cfg.training["num_workers"] = 0  # same worker-crash fix as 02_ecg_training.ipynb

loaders = build_dataloaders(waveforms, labels, splits, cfg)

# The 02 notebook's CNN is OPTIONAL here: v2 trains its own, stronger signal
# branch in §11. If the old checkpoint is found it is added as one more input
# to the stacked ensemble; if not, the notebook carries on without it.
CHECKPOINTS = MODELS / "checkpoints"   # (was undefined in v1 -> NameError)
DRIVE_ECG = Path("/content/drive/MyDrive/CardioSense") / "models" / "ecg"

candidate_paths = [
    MODELS / cfg.output.model_file,
    CHECKPOINTS / "best.pt",
    CHECKPOINTS / str(cfg.model.name) / "best.pt",
    DRIVE_ECG / cfg.output.model_file,
    DRIVE_ECG / "checkpoints" / "best.pt",
    DRIVE_ECG / "checkpoints" / str(cfg.model.name) / "best.pt",
]

def torch_load(path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:  # older torch without weights_only
        return torch.load(path, map_location=device)

signal_model, loaded_from, loaded_thresholds = None, None, None
for path in candidate_paths:
    if path.exists():
        ckpt = torch_load(path)
        model = build_model(cfg).to(device)
        state = ckpt.get("model_state", ckpt.get("state_dict", ckpt)) if isinstance(ckpt, dict) else ckpt
        model.load_state_dict(state)
        signal_model, loaded_from = model.eval(), path
        if isinstance(ckpt, dict) and "thresholds" in ckpt:
            loaded_thresholds = np.asarray(ckpt["thresholds"])
        break

_sanity_metrics = None
if signal_model is None:
    print("Old 02-notebook CNN not found (tried:\n  " + "\n  ".join(map(str, candidate_paths)) +
          ")\n-> continuing without it; §11 trains a stronger signal branch anyway.")
else:
    print(f"loaded old CNN from: {loaded_from} ({count_parameters(signal_model):,} params)")
    cnn_val_prob_full, _ = ev.predict_probabilities(signal_model, loaders["val"], device, use_amp)
    cnn_test_prob_full, cnn_test_true_full = ev.predict_probabilities(signal_model, loaders["test"], device, use_amp)
    _thr, _ = ev.tune_per_class_thresholds(splits["val"]["labels"], cnn_val_prob_full, cfg)
    _sanity_metrics = ev.evaluate_multilabel(cnn_test_true_full, cnn_test_prob_full, _thr, cfg,
                                             split_name="test/old-cnn-full", with_ci=False)
    print(f"old CNN, full test: macro F1 {_sanity_metrics['macro_f1']:.4f}, "
          f"macro AUC {_sanity_metrics['macro_roc_auc']:.4f}")

## 6. Download PTB-XL+ features

Open access, no credentialing. Only the two feature CSVs the paper used are
downloaded — **not** the full 2.0 GB PTB-XL+ release (median beats + fiducial
points, which this notebook doesn't need):

| File | Size |
|---|---|
| `12sl_features.csv` | 88.9 MB |
| `unig_features.csv` | 87.9 MB |

Source: https://physionet.org/content/ptb-xl-plus/1.0.1/features/

In [ ]:
import urllib.request

PTBXL_PLUS_DIR = Path(os.environ["CARDIOSENSE_DATA_ROOT"]) / "ptbxl_plus"
PTBXL_PLUS_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_FILES = {
    "12sl": "12sl_features.csv",
    "unig": "unig_features.csv",
}
BASE_URL = "https://physionet.org/files/ptb-xl-plus/1.0.1/features/"

for key, fname in FEATURE_FILES.items():
    dest = PTBXL_PLUS_DIR / fname
    if dest.exists() and dest.stat().st_size > 1_000_000:
        print(f"{fname}: already present ({dest.stat().st_size / 1024**2:.1f} MB), skipping")
        continue
    url = BASE_URL + fname + "?download"
    print(f"downloading {fname} ...")
    urllib.request.urlretrieve(url, dest)
    print(f"  saved {dest} ({dest.stat().st_size / 1024**2:.1f} MB)")

## 7. Join features to splits, on `ecg_id`

Both CSVs ship one row per PTB-XL record with a PTB-XL ECG-identifier column.
Column name isn't assumed — detected against `database.index.name` (`ecg_id`)
defensively in case it differs slightly between the two files.

Some records fail feature extraction entirely (all-NaN row) — those are
**dropped from this experiment only** (features-only branch and fusion),
not from the signal-only CNN's own numbers. The drop count is reported so
that's visible, not silently absorbed. Individual missing *values* (not a
whole row) are median-imputed in §8, fit on train only.

In [ ]:
def load_feature_csv(path, id_col_hint="ecg_id"):
    df = pd.read_csv(path)
    id_col = None
    for c in df.columns:
        if c.strip().lower() == id_col_hint:
            id_col = c
            break
    if id_col is None:
        # fall back: first integer-like column
        id_col = df.columns[0]
        print(f"  WARNING: no exact '{id_col_hint}' column found in {path.name}, "
              f"falling back to first column '{id_col}' — verify this is the ECG id.")
    df = df.rename(columns={id_col: "ecg_id"}).set_index("ecg_id")
    df.columns = [f"{path.stem.split('_')[0]}__{c}" for c in df.columns]
    return df

feat_12sl = load_feature_csv(PTBXL_PLUS_DIR / "12sl_features.csv")
feat_unig = load_feature_csv(PTBXL_PLUS_DIR / "unig_features.csv")

print(f"12sl : {feat_12sl.shape[0]} rows, {feat_12sl.shape[1]} features")
print(f"unig : {feat_unig.shape[0]} rows, {feat_unig.shape[1]} features")

# outer-join the two feature sets on ecg_id (paper's "combined Uni-G/12SL")
features_all = feat_12sl.join(feat_unig, how="outer")
print(f"combined: {features_all.shape[0]} rows, {features_all.shape[1]} features")

# keep only numeric feature columns
feature_cols = features_all.select_dtypes(include=[np.number]).columns.tolist()
features_all = features_all[feature_cols]
print(f"numeric feature columns kept: {len(feature_cols)}")

In [ ]:
def align_split(split_name):
    """Reindex features to this split's record order; report + drop all-NaN rows."""
    split_db = splits[split_name]["database"]
    y = splits[split_name]["labels"]

    X = features_all.reindex(split_db.index)
    all_nan = X.isna().all(axis=1).to_numpy()
    n_drop = int(all_nan.sum())
    if n_drop:
        print(f"  {split_name}: dropping {n_drop} / {len(split_db)} records "
              f"({100*n_drop/len(split_db):.1f}%) with no feature-extraction match")

    keep = ~all_nan
    return X.loc[keep], y[keep], keep

X_train_raw, y_train_feat, keep_train = align_split("train")
X_val_raw, y_val_feat, keep_val = align_split("val")
X_test_raw, y_test_feat, keep_test = align_split("test")

print(f"\nusable records — train: {len(X_train_raw)}, val: {len(X_val_raw)}, "
      f"test: {len(X_test_raw)}")

### 7b. Demographics + column cleaning (train-fit only)

Age and sex change what "normal" looks like on an ECG (voltage criteria for
HYP are explicitly age/sex dependent), and they are in `ptbxl_database.csv`
for every record — no extra data needed. Toggle `USE_DEMOGRAPHICS=False` to
reproduce the paper's pure Uni-G/12SL feature set as an ablation.

Columns that are >95% missing or constant **on train** are dropped; ±inf
values (occasional division artefacts in the extractors) become NaN.

In [ ]:
# USE_DEMOGRAPHICS is set in the run-settings cell at the top of the notebook

def demographics(split_name, keep):
    db = splits[split_name]["database"][keep]
    out = pd.DataFrame(index=db.index)
    age = pd.to_numeric(db["age"], errors="coerce").astype(float)
    out["demo__age"] = age.where(age < 300, 90.0)   # PTB-XL codes age>89 as 300
    out["demo__sex"] = pd.to_numeric(db["sex"], errors="coerce").astype(float)
    for col in ("height", "weight"):
        if col in db.columns:
            out[f"demo__{col}"] = pd.to_numeric(db[col], errors="coerce").astype(float)
    return out

if USE_DEMOGRAPHICS:
    X_train_raw = X_train_raw.join(demographics("train", keep_train))
    X_val_raw = X_val_raw.join(demographics("val", keep_val))
    X_test_raw = X_test_raw.join(demographics("test", keep_test))

for _df in (X_train_raw, X_val_raw, X_test_raw):
    _df.replace([np.inf, -np.inf], np.nan, inplace=True)

train_missing = X_train_raw.isna().mean()
train_nunique = X_train_raw.nunique(dropna=True)
drop_cols = train_missing.index[(train_missing > 0.95) | (train_nunique <= 1)]
X_train_raw = X_train_raw.drop(columns=drop_cols)
X_val_raw = X_val_raw.drop(columns=drop_cols)
X_test_raw = X_test_raw.drop(columns=drop_cols)
FEATURE_NAMES = X_train_raw.columns.tolist()

print(f"dropped {len(drop_cols)} empty/constant columns; {len(FEATURE_NAMES)} features kept "
      f"(demographics: {USE_DEMOGRAPHICS})")

## 8. Preprocess features — impute + scale + clip, fit on train only

Median imputation, standard scaling, then clipping to ±5 SD — a handful of
extreme extractor outputs otherwise dominate the MLP's first layer. All three
are fit on **train only**. XGBoost (§10) uses the raw, un-imputed values
instead, since it learns its own missing-value routing.

In [ ]:
CLIP_SD = 5.0
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train = np.clip(scaler.fit_transform(imputer.fit_transform(X_train_raw)), -CLIP_SD, CLIP_SD).astype(np.float32)
X_val = np.clip(scaler.transform(imputer.transform(X_val_raw)), -CLIP_SD, CLIP_SD).astype(np.float32)
X_test = np.clip(scaler.transform(imputer.transform(X_test_raw)), -CLIP_SD, CLIP_SD).astype(np.float32)

y_train_feat = np.asarray(y_train_feat, dtype=np.float32)
y_val_feat = np.asarray(y_val_feat, dtype=np.float32)
y_test_feat = np.asarray(y_test_feat, dtype=np.float32)

n_classes, n_feat = len(CLASSES), X_train.shape[1]
print(f"feature matrix: train {X_train.shape}, val {X_val.shape}, test {X_test.shape}")

### Shared helpers — one evaluation path for every experiment

Every model goes through `register()`: thresholds tuned per class on **val**,
one pass over **test**. `val_store` / `test_store` keep each branch's
probabilities for the ensembles in §12.

In [ ]:
from sklearn.metrics import roc_auc_score

EPS = 1e-6
# N_SEEDS is set in the run-settings cell at the top of the notebook
SEEDS = [SEED + i for i in range(N_SEEDS)]

def to_logit(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def mean_logit(probs):
    return sigmoid(np.mean([to_logit(p) for p in probs], axis=0))

def macro_auc(y, p):
    return float(np.mean([roc_auc_score(y[:, c], p[:, c]) for c in range(y.shape[1])]))

experiments, val_store, test_store = {}, {}, {}

def register(name, p_val, p_test, y_val=None, y_test=None, params=None, secs=None, store=True):
    y_val = y_val_feat if y_val is None else y_val
    y_test = y_test_feat if y_test is None else y_test
    thr, _ = ev.tune_per_class_thresholds(y_val, p_val, cfg)
    m = ev.evaluate_multilabel(y_test, p_test, thr, cfg, split_name=f"test/{name}", with_ci=False)
    experiments[name] = {"metrics": m, "parameters": params, "train_seconds": secs,
                         "val_macro_auc": macro_auc(y_val, p_val), "thresholds": thr}
    if store:
        val_store[name], test_store[name] = p_val, p_test
    print(f">> {name:<26} val AUC {experiments[name]['val_macro_auc']:.4f} | "
          f"test macro F1 {m['macro_f1']:.4f} | test macro AUC {m['macro_roc_auc']:.4f}")
    return m

pos_weight = compute_pos_weight(y_train_feat).clamp(max=10.0).to(device)
print("pos_weight:", {c: round(float(w), 2) for c, w in zip(CLASSES, pos_weight.tolist())})

if signal_model is not None:   # old CNN on the feature subset, as a reference + stacking input
    register("signal_cnn_v1", cnn_val_prob_full[keep_val], cnn_test_prob_full[keep_test],
             params=count_parameters(signal_model))

## 9. Features-only dense network — Experiment E-D1

Wider MLP with BatchNorm + SiLU, AdamW + cosine schedule, light Gaussian
input noise as regularisation, early stopping on val macro AUC. Oversampling
from v1 is dropped: combined with `pos_weight` it double-counts the rare
classes and only shifts calibration (thresholds are re-tuned anyway).
Averaged over `N_SEEDS` seeds.

In [ ]:
class FeatureMLP(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=(512, 256), dropout=0.3):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.SiLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

Xtr_t, Ytr_t = torch.as_tensor(X_train), torch.as_tensor(y_train_feat)
Xva_t, Xte_t = torch.as_tensor(X_val), torch.as_tensor(X_test)

@torch.no_grad()
def mlp_predict_proba(model, X_t):
    model.eval()
    return torch.sigmoid(model(X_t.to(device))).cpu().numpy()

def train_feature_mlp(seed, max_epochs=120, patience=15, lr=1e-3, noise=0.05, bs=256):
    set_seed(seed)
    model = FeatureMLP(n_feat, n_classes).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    gen = torch.Generator().manual_seed(seed)
    best, best_state, bad = -1.0, None, 0
    for epoch in range(max_epochs):
        model.train()
        for idx in torch.randperm(len(Xtr_t), generator=gen).split(bs):
            xb, yb = Xtr_t[idx].to(device), Ytr_t[idx].to(device)
            xb = xb + noise * torch.randn_like(xb)
            opt.zero_grad()
            crit(model(xb), yb).backward()
            opt.step()
        sched.step()
        auc = macro_auc(y_val_feat, mlp_predict_proba(model, Xva_t))
        if auc > best + 1e-4:
            best, bad = auc, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    print(f"  seed {seed}: best val macro AUC {best:.4f} (stopped at epoch {epoch + 1})")
    return model

start = time.time()
mlp_models = [train_feature_mlp(s) for s in SEEDS]
mlp_seconds = time.time() - start

val_prob_mlp = mean_logit([mlp_predict_proba(m, Xva_t) for m in mlp_models])
test_prob_mlp = mean_logit([mlp_predict_proba(m, Xte_t) for m in mlp_models])
register("features_mlp", val_prob_mlp, test_prob_mlp,
         params=count_parameters(mlp_models[0]) * N_SEEDS, secs=mlp_seconds)

## 10. Features-only gradient boosting — Experiment E-D2 / E-D3

One binary XGBoost per class (one-vs-rest), on the **raw** features (NaN kept
— XGBoost learns a default direction for missing values). Early stopping on
val AUC. Runs on the GPU when available (~1–3 min total on a T4).

E-D3 averages the MLP and XGBoost in logit space: the two learners make
different kinds of errors, so their average is usually better than either.

In [ ]:
try:
    import xgboost as xgb
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost>=2.0"], check=True)
    import xgboost as xgb

XGB_PARAMS = dict(
    n_estimators=4000, learning_rate=0.02, max_depth=6, min_child_weight=2,
    subsample=0.8, colsample_bytree=0.4, reg_lambda=2.0, tree_method="hist",
    eval_metric="auc", early_stopping_rounds=200, random_state=SEED,
)
if int(xgb.__version__.split(".")[0]) >= 2:
    XGB_PARAMS["device"] = "cuda" if device.type == "cuda" else "cpu"

Xtr_raw = X_train_raw.to_numpy(np.float32)
Xva_raw = X_val_raw.to_numpy(np.float32)
Xte_raw = X_test_raw.to_numpy(np.float32)

start = time.time()
xgb_models = []
val_prob_xgb = np.zeros_like(y_val_feat)
test_prob_xgb = np.zeros_like(y_test_feat)
for c, name in enumerate(CLASSES):
    clf = xgb.XGBClassifier(**XGB_PARAMS)
    clf.fit(Xtr_raw, y_train_feat[:, c], eval_set=[(Xva_raw, y_val_feat[:, c])], verbose=False)
    val_prob_xgb[:, c] = clf.predict_proba(Xva_raw)[:, 1]
    test_prob_xgb[:, c] = clf.predict_proba(Xte_raw)[:, 1]
    xgb_models.append(clf)
    print(f"  {name:5s}: best iteration {clf.best_iteration:4d} | val AUC "
          f"{roc_auc_score(y_val_feat[:, c], val_prob_xgb[:, c]):.4f}")
xgb_seconds = time.time() - start

register("features_xgb", val_prob_xgb, test_prob_xgb, secs=xgb_seconds)
register("features_ensemble", mean_logit([val_prob_mlp, val_prob_xgb]),
         mean_logit([test_prob_mlp, test_prob_xgb]), secs=mlp_seconds + xgb_seconds)

# top features by gain, averaged over the 5 class models — useful for the write-up
gain = pd.DataFrame({n: pd.Series(m.get_booster().get_score(importance_type="gain"))
                     for n, m in zip(CLASSES, xgb_models)}).fillna(0)
gain.index = [FEATURE_NAMES[int(i[1:])] if i.startswith("f") and i[1:].isdigit() else i for i in gain.index]
display(gain.assign(mean=gain.mean(axis=1)).sort_values("mean", ascending=False).head(15).round(1))

## 11. Gated joint network — Experiments E-J (fused) and E-S (signal-only)

The paper's combined model gave only marginal gains. Three design choices
here target the usual reasons naive concatenation-fusion fails:

1. **A stronger signal encoder** — SE-ResNet-1D (residual blocks with
   squeeze-and-excitation channel attention, kernel 9 ≈ 90 ms at 100 Hz),
   trained with OneCycle LR. The v1 plain CNN was the weak link
   (0.909 AUC vs the paper's 0.93).
2. **Learned gated fusion** — a per-dimension sigmoid gate
   `g = σ(W[z_sig; z_feat])` mixes the two 256-d embeddings, so the network
   can lean on features for one pattern and on the raw signal for another.
   The mean gate value is reported per class below — it is also the input to
   the project's explanation-faithfulness analysis.
3. **Modality dropout + deep supervision** — during training, one modality's
   embedding is zeroed for 15% of samples each, and each encoder also gets its
   own auxiliary classification head (loss weight 0.5). This stops the fused
   head from relying only on the easier feature branch (the classic
   "modality laziness" failure) and makes the signal encoder a **standalone
   signal-only classifier** (E-S) at no extra training cost.

Waveform augmentation (train only): ±10% amplitude, random circular shift
up to 1 s, Gaussian noise σ=0.03. Checkpoints go to Drive **every epoch** and
training resumes automatically after a Colab disconnect.

In [ ]:
class SE1d(nn.Module):
    def __init__(self, c, r=8):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(c, c // r), nn.ReLU(inplace=True), nn.Linear(c // r, c), nn.Sigmoid())

    def forward(self, x):
        return x * self.fc(x.mean(-1)).unsqueeze(-1)

class ResBlockSE(nn.Module):
    def __init__(self, cin, cout, stride=1, k=9, dropout=0.2):
        super().__init__()
        self.conv1 = nn.Conv1d(cin, cout, k, stride=stride, padding=k // 2, bias=False)
        self.bn1 = nn.BatchNorm1d(cout)
        self.conv2 = nn.Conv1d(cout, cout, k, padding=k // 2, bias=False)
        self.bn2 = nn.BatchNorm1d(cout)
        self.se = SE1d(cout)
        self.drop = nn.Dropout(dropout)
        self.short = (nn.Identity() if stride == 1 and cin == cout else
                      nn.Sequential(nn.Conv1d(cin, cout, 1, stride=stride, bias=False), nn.BatchNorm1d(cout)))

    def forward(self, x):
        out = self.drop(torch.relu(self.bn1(self.conv1(x))))
        out = self.se(self.bn2(self.conv2(out)))
        return torch.relu(out + self.short(x))

class SignalEncoder(nn.Module):
    """SE-ResNet-1D: (B, 12, 1000) -> (B, 2 * widths[-1]) via global avg+max pool."""
    def __init__(self, in_ch=12, widths=(64, 128, 192, 256), blocks=2, k=9, dropout=0.2):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(in_ch, widths[0], 15, stride=2, padding=7, bias=False),
                                  nn.BatchNorm1d(widths[0]), nn.ReLU(inplace=True))
        layers, prev = [], widths[0]
        for s, w in enumerate(widths):
            for b in range(blocks):
                layers.append(ResBlockSE(prev, w, stride=2 if (b == 0 and s > 0) else 1, k=k, dropout=dropout))
                prev = w
        self.stages = nn.Sequential(*layers)
        self.out_dim = prev * 2

    def forward(self, x):
        x = self.stages(self.stem(x))
        return torch.cat([x.mean(-1), x.amax(-1)], dim=1)

class GatedJointNet(nn.Module):
    def __init__(self, n_feat, n_classes, d=256, p_moddrop=0.15):
        super().__init__()
        self.p_moddrop = p_moddrop
        self.sig_enc = SignalEncoder()
        self.sig_proj = nn.Sequential(nn.Linear(self.sig_enc.out_dim, d), nn.BatchNorm1d(d), nn.SiLU())
        self.feat_enc = nn.Sequential(nn.Linear(n_feat, 512), nn.BatchNorm1d(512), nn.SiLU(), nn.Dropout(0.3),
                                      nn.Linear(512, d), nn.BatchNorm1d(d), nn.SiLU())
        self.gate = nn.Sequential(nn.Linear(2 * d, d), nn.Sigmoid())
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(3 * d, d), nn.SiLU(), nn.Dropout(0.2),
                                  nn.Linear(d, n_classes))
        self.head_sig = nn.Sequential(nn.Dropout(0.3), nn.Linear(d, n_classes))
        self.head_feat = nn.Sequential(nn.Dropout(0.3), nn.Linear(d, n_classes))

    def encode_signal(self, w):
        return self.sig_proj(self.sig_enc(w))

    def signal_logits(self, w):
        return self.head_sig(self.encode_signal(w))

    def forward(self, w, f):
        zs, zf = self.encode_signal(w), self.feat_enc(f)
        zs_in, zf_in = zs, zf
        if self.training and self.p_moddrop > 0:
            r = torch.rand(zs.shape[0], 1, device=zs.device)
            zs_in = zs.masked_fill(r < self.p_moddrop, 0.0)
            zf_in = zf.masked_fill((r >= self.p_moddrop) & (r < 2 * self.p_moddrop), 0.0)
        g = self.gate(torch.cat([zs_in, zf_in], dim=1))
        fused = g * zs_in + (1 - g) * zf_in
        logits = self.head(torch.cat([zs_in, zf_in, fused], dim=1))
        return logits, self.head_sig(zs), self.head_feat(zf), g

_probe = GatedJointNet(n_feat, n_classes)
print(f"GatedJointNet: {count_parameters(_probe):,} params "
      f"(signal encoder {count_parameters(_probe.sig_enc):,})")
del _probe

In [ ]:
# ── tensors: feature subset for joint training; FULL val/test for the signal-only head
def waveform_tensor(split_name, keep=None):
    idx = splits[split_name]["indices"]
    if keep is not None:
        idx = idx[keep]
    return torch.from_numpy(np.ascontiguousarray(np.asarray(waveforms[idx], dtype=np.float32)))

Wtr, Wva, Wte = waveform_tensor("train", keep_train), waveform_tensor("val", keep_val), waveform_tensor("test", keep_test)
Wva_full, Wte_full = waveform_tensor("val"), waveform_tensor("test")
y_val_full = np.asarray(splits["val"]["labels"], dtype=np.float32)
y_test_full = np.asarray(splits["test"]["labels"], dtype=np.float32)
print("waveforms:", tuple(Wtr.shape), tuple(Wva.shape), tuple(Wte.shape), "| full val/test:",
      tuple(Wva_full.shape), tuple(Wte_full.shape))

JOINT = dict(epochs=45, batch_size=128, lr=1e-3, weight_decay=1e-2, aux_weight=0.5,
             patience=12, max_shift=100, amp_scale=0.1, noise=0.03, feat_noise=0.05)

FUSION_CKPT_DIR = (Path("/content/drive/MyDrive/CardioSense/models/ecg_fusion") if IN_COLAB
                   else MODELS.parent / "ecg_fusion")
if not USE_DEMOGRAPHICS:   # different feature count -> keep its checkpoints apart
    FUSION_CKPT_DIR = FUSION_CKPT_DIR / "no_demographics"
FUSION_CKPT_DIR.mkdir(parents=True, exist_ok=True)

def augment_waveforms(w):
    B, C, L = w.shape
    w = w * (1 + (torch.rand(B, 1, 1, device=w.device) * 2 - 1) * JOINT["amp_scale"])
    shift = torch.randint(-JOINT["max_shift"], JOINT["max_shift"] + 1, (B, 1), device=w.device)
    idx = (torch.arange(L, device=w.device)[None, :] - shift) % L
    w = w.gather(2, idx[:, None, :].expand(B, C, L))
    return w + JOINT["noise"] * torch.randn_like(w)

@torch.no_grad()
def joint_predict(model, W, F, bs=512):
    model.eval()
    fused, sig, feat, gates = [], [], [], []
    for i in range(0, len(W), bs):
        w, f = W[i:i + bs].to(device), F[i:i + bs].to(device)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            lo, ls, lf, g = model(w, f)
        fused.append(torch.sigmoid(lo.float()).cpu()); sig.append(torch.sigmoid(ls.float()).cpu())
        feat.append(torch.sigmoid(lf.float()).cpu()); gates.append(g.float().mean(1).cpu())
    return tuple(torch.cat(t).numpy() for t in (fused, sig, feat, gates))

@torch.no_grad()
def signal_predict(model, W, bs=512):
    model.eval()
    out = []
    for i in range(0, len(W), bs):
        with torch.autocast(device_type=device.type, enabled=use_amp):
            out.append(torch.sigmoid(model.signal_logits(W[i:i + bs].to(device)).float()).cpu())
    return torch.cat(out).numpy()

def train_joint(seed):
    """Train one seed; resumes from Drive after a disconnect, skips if already finished."""
    ckpt_path = FUSION_CKPT_DIR / f"joint_seed{seed}.pt"
    set_seed(seed)
    model = GatedJointNet(n_feat, n_classes).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=JOINT["lr"], weight_decay=JOINT["weight_decay"])
    steps = (len(Wtr) + JOINT["batch_size"] - 1) // JOINT["batch_size"]
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=JOINT["lr"], epochs=JOINT["epochs"],
                                                steps_per_epoch=steps, pct_start=0.25)
    amp_scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    st = {"epoch": 0, "best_auc": -1.0, "best_state": None, "bad": 0, "done": False,
          "n_feat": n_feat, "feature_names": FEATURE_NAMES}

    if ckpt_path.exists():
        saved = torch_load(ckpt_path)
        if saved["state"]["n_feat"] == n_feat:
            model.load_state_dict(saved["model"]); opt.load_state_dict(saved["opt"])
            sched.load_state_dict(saved["sched"]); amp_scaler.load_state_dict(saved["scaler"])
            st = saved["state"]
            print(f"  seed {seed}: resumed at epoch {st['epoch']} (best val AUC {st['best_auc']:.4f})"
                  + (" — already finished" if st["done"] else ""))
        else:
            print(f"  seed {seed}: checkpoint feature count differs (config changed) — retraining")

    gen = torch.Generator().manual_seed(seed)
    for _ in range(st["epoch"]):          # keep the shuffle stream identical after a resume
        torch.randperm(len(Wtr), generator=gen)

    t0 = time.time()
    while not st["done"] and st["epoch"] < JOINT["epochs"]:
        model.train()
        run_loss = 0.0
        for idx in torch.randperm(len(Wtr), generator=gen).split(JOINT["batch_size"]):
            w = augment_waveforms(Wtr[idx].to(device, non_blocking=True))
            f = Xtr_t[idx].to(device)
            f = f + JOINT["feat_noise"] * torch.randn_like(f)
            y = Ytr_t[idx].to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                lo, ls, lf, _ = model(w, f)
                loss = crit(lo.float(), y) + JOINT["aux_weight"] * (crit(ls.float(), y) + crit(lf.float(), y))
            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            amp_scaler.step(opt); amp_scaler.update(); sched.step()
            run_loss += loss.item() * len(idx)

        pv, ps, _, _ = joint_predict(model, Wva, Xva_t)
        auc, auc_sig = macro_auc(y_val_feat, pv), macro_auc(y_val_feat, ps)
        st["epoch"] += 1
        mark = ""
        if auc > st["best_auc"] + 1e-4:
            st.update(best_auc=auc, bad=0,
                      best_state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
            mark = "  <- best"
        else:
            st["bad"] += 1
        if st["bad"] >= JOINT["patience"] or st["epoch"] >= JOINT["epochs"]:
            st["done"] = True
        print(f"  seed {seed} | epoch {st['epoch']:2d}/{JOINT['epochs']} | loss {run_loss / len(Wtr):.4f} | "
              f"val AUC fused {auc:.4f} signal-head {auc_sig:.4f}{mark}")
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": amp_scaler.state_dict(), "state": st}, ckpt_path)

    model.load_state_dict(st["best_state"])
    return model, time.time() - t0

In [ ]:
start = time.time()
joint_models = []
for s in SEEDS:
    m, _ = train_joint(s)
    joint_models.append(m)
joint_seconds = time.time() - start

val_out = [joint_predict(m, Wva, Xva_t) for m in joint_models]
test_out = [joint_predict(m, Wte, Xte_t) for m in joint_models]
n_joint_params = count_parameters(joint_models[0]) * N_SEEDS

register("joint_gated", mean_logit([o[0] for o in val_out]), mean_logit([o[0] for o in test_out]),
         params=n_joint_params, secs=joint_seconds)
register("signal_seresnet", mean_logit([o[1] for o in val_out]), mean_logit([o[1] for o in test_out]),
         params=count_parameters(joint_models[0].sig_enc) * N_SEEDS)
register("joint_feature_head", mean_logit([o[2] for o in val_out]), mean_logit([o[2] for o in test_out]),
         store=False)

# signal-only head on the FULL val/test (incl. records with no PTB-XL+ features) —
# directly comparable to the paper's signal-only row and to 02_ecg_training.ipynb
register("signal_seresnet_fulltest",
         mean_logit([signal_predict(m, Wva_full) for m in joint_models]),
         mean_logit([signal_predict(m, Wte_full) for m in joint_models]),
         y_val=y_val_full, y_test=y_test_full, store=False)

# mean gate value = how much the fused embedding leans on the SIGNAL branch (1.0 = all signal)
gate_test = np.mean([o[3] for o in test_out], axis=0)
print("\nmean fusion gate toward signal, by true class (test):")
for c, name in enumerate(CLASSES):
    print(f"  {name:5s}: {gate_test[y_test_feat[:, c] == 1].mean():.3f}")

## 12. Ensembles — E-E, E-M, E-K

* **E-E  AUC-weighted log-odds** (v1's fusion, kept for continuity): best
  signal branch + feature ensemble, weighted per class by val AUC.
* **E-M  mean-logit ensemble** of joint + signal + XGB + MLP.
* **E-K  stacked meta-learner**: per class, a logistic regression over the
  logits of *every* branch for *all five* classes (cross-class evidence helps
  — a confident NORM argues against MI). It is fit on **val**; thresholds are
  tuned on 5-fold **out-of-fold** val predictions, so neither the meta-weights
  nor the thresholds ever see test.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

# E-E: v1's AUC-weighted log-odds fusion, now on the stronger branches
w_sig = np.array([roc_auc_score(y_val_feat[:, c], val_store["signal_seresnet"][:, c]) for c in range(n_classes)])
w_feat = np.array([roc_auc_score(y_val_feat[:, c], val_store["features_ensemble"][:, c]) for c in range(n_classes)])
def auc_weighted(ps, pf):
    return sigmoid((w_sig * to_logit(ps) + w_feat * to_logit(pf)) / (w_sig + w_feat))
register("fused_logodds_auc", auc_weighted(val_store["signal_seresnet"], val_store["features_ensemble"]),
         auc_weighted(test_store["signal_seresnet"], test_store["features_ensemble"]))

# E-M: plain mean of logits
MEAN_MEMBERS = ["joint_gated", "signal_seresnet", "features_xgb", "features_mlp"]
register("ensemble_mean", mean_logit([val_store[k] for k in MEAN_MEMBERS]),
         mean_logit([test_store[k] for k in MEAN_MEMBERS]))

# E-K: stacking
STACK_MEMBERS = ["joint_gated", "signal_seresnet", "features_xgb", "features_mlp"] + \
                (["signal_cnn_v1"] if "signal_cnn_v1" in val_store else [])
Zv = np.hstack([to_logit(val_store[k]) for k in STACK_MEMBERS])
Zt = np.hstack([to_logit(test_store[k]) for k in STACK_MEMBERS])
mu, sd = Zv.mean(0), Zv.std(0) + 1e-6
Zv, Zt = (Zv - mu) / sd, (Zt - mu) / sd

oof_val = np.zeros_like(y_val_feat)
test_prob_stack = np.zeros_like(y_test_feat)
for c in range(n_classes):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    for tr_i, va_i in skf.split(Zv, y_val_feat[:, c]):
        lr = LogisticRegression(C=0.3, max_iter=5000).fit(Zv[tr_i], y_val_feat[tr_i, c])
        oof_val[va_i, c] = lr.predict_proba(Zv[va_i])[:, 1]
    lr = LogisticRegression(C=0.3, max_iter=5000).fit(Zv, y_val_feat[:, c])
    test_prob_stack[:, c] = lr.predict_proba(Zt)[:, 1]
print("stack members:", STACK_MEMBERS)
register("stacked_lr", oof_val, test_prob_stack)

## 13. Comparison — selected by validation, reported on test

The **final model is the one with the highest validation macro AUC** among the
feature-subset models. Test numbers are shown for every row, but they play no
part in the choice. (Caveat: E-D/E-J early-stop on val, so their val AUCs are
slightly optimistic relative to E-K's out-of-fold val AUC — which biases the
selection *against* stacking, not towards it.)

In [ ]:
comparison = ev.build_comparison_table(experiments)
comparison.insert(1, "val_macro_auc", [round(experiments[k]["val_macro_auc"], 4) for k in comparison["model"]])
comparison.to_csv(FUSION_RESULTS / "fusion_comparison.csv", index=False)
display(comparison.sort_values("val_macro_auc", ascending=False))

candidates = {k: v for k, v in experiments.items() if not k.endswith("fulltest")}
FINAL = max(candidates, key=lambda k: candidates[k]["val_macro_auc"])
print(f"\nSELECTED BY VALIDATION: {FINAL}")

# full metrics with bootstrap CIs for the selected model only
final_metrics = ev.evaluate_multilabel(y_test_feat, test_store[FINAL], experiments[FINAL]["thresholds"],
                                       cfg, split_name=f"test/{FINAL}", with_ci=True)
experiments[FINAL]["metrics"] = final_metrics
display(ev.per_class_table(final_metrics, CLASSES))

save_json({"selected": FINAL, "selected_by": "val_macro_auc",
           "val_macro_auc": {k: round(v["val_macro_auc"], 4) for k, v in experiments.items()},
           "thresholds": dict(zip(CLASSES, np.round(experiments[FINAL]["thresholds"], 4).tolist())),
           "use_demographics": USE_DEMOGRAPHICS, "n_seeds": N_SEEDS, "n_features": n_feat},
          FUSION_RESULTS / "fusion_recommendation.json")
np.savez_compressed(FUSION_RESULTS / "test_probabilities.npz", y_test=y_test_feat,
                    ecg_id=X_test_raw.index.to_numpy(), **test_store)

### 13b. Confidence intervals on the macro scores

The test set is resampled 1,000 times (records drawn with replacement). Each model's
thresholds stay fixed at their validation-tuned values, so only the test sample varies.
This shows how much a score could move with a different 2,158-record test set, and
whether the final model's lead over features-only XGBoost is more than noise
(a paired difference whose interval contains 0 is within noise).

In [ ]:
from sklearn.metrics import f1_score

N_BOOT = 1000
rng = np.random.default_rng(SEED)
boot_idx = [rng.integers(0, len(y_test_feat), len(y_test_feat)) for _ in range(N_BOOT)]

def macro_scores(y, p, thr):
    aucs = [roc_auc_score(y[:, c], p[:, c]) for c in range(y.shape[1]) if 0 < y[:, c].sum() < len(y)]
    f1 = f1_score(y, (p >= thr[None, :]).astype(int), average="macro", zero_division=0)
    return float(np.mean(aucs)), float(f1)

def bootstrap(name):
    p, thr = test_store[name], np.asarray(experiments[name]["thresholds"])
    return np.array([macro_scores(y_test_feat[i], p[i], thr) for i in boot_idx])   # (N_BOOT, 2)

CI_MODELS = list(dict.fromkeys(m for m in [FINAL, "features_xgb", "joint_gated", "signal_seresnet", "features_mlp"]
                               if m in test_store))
start = time.time()
boots = {m: bootstrap(m) for m in CI_MODELS}

rows = []
for m, b in boots.items():
    auc, f1 = macro_scores(y_test_feat, test_store[m], np.asarray(experiments[m]["thresholds"]))
    lo, hi = np.percentile(b, [2.5, 97.5], axis=0)
    rows.append({"model": m,
                 "macro_auc": round(auc, 4), "auc_95ci": f"[{lo[0]:.4f}, {hi[0]:.4f}]",
                 "macro_f1": round(f1, 4), "f1_95ci": f"[{lo[1]:.4f}, {hi[1]:.4f}]",
                 "resamples_auc_above_0.93": round(float((b[:, 0] > 0.93).mean()), 3),
                 "resamples_f1_above_0.75": round(float((b[:, 1] > 0.75).mean()), 3)})
ci_table = pd.DataFrame(rows)
ci_table.to_csv(FUSION_RESULTS / "bootstrap_ci.csv", index=False)
display(ci_table)
print(f"({N_BOOT} resamples in {time.time() - start:.0f}s)")

paired = None
if FINAL != "features_xgb" and "features_xgb" in boots:
    d = boots[FINAL] - boots["features_xgb"]
    lo, hi = np.percentile(d, [2.5, 97.5], axis=0)
    paired = {"comparison": f"{FINAL} minus features_xgb",
              "auc_diff": round(float(d[:, 0].mean()), 4), "auc_diff_95ci": [round(float(lo[0]), 4), round(float(hi[0]), 4)],
              "f1_diff": round(float(d[:, 1].mean()), 4), "f1_diff_95ci": [round(float(lo[1]), 4), round(float(hi[1]), 4)]}
    print(f"\n{paired['comparison']}: AUC {paired['auc_diff']:+.4f} {paired['auc_diff_95ci']} | "
          f"F1 {paired['f1_diff']:+.4f} {paired['f1_diff_95ci']}")
    print("An interval that contains 0 means the difference is within sampling noise.")

## 14. Benchmark check — against the published paper

In [ ]:
PAPER = {"paper features-only": (0.73, 0.92), "paper signal-only": (0.75, 0.93)}

rows = []
if _sanity_metrics is not None:
    rows.append(("v1 CNN (full test, N=%d)" % len(y_test_full), _sanity_metrics["macro_f1"], _sanity_metrics["macro_roc_auc"]))
for name, block in experiments.items():
    n = len(y_test_full) if name.endswith("fulltest") else len(y_test_feat)
    rows.append((f"{name} (N={n})", block["metrics"]["macro_f1"], block["metrics"]["macro_roc_auc"]))

print(f"{'run':<42}{'macro F1':<11}{'macro AUC':<11}beats paper signal-only?")
for name, f1, auc in rows:
    beats = "YES" if (f1 > 0.75 and auc > 0.93) else ""
    print(f"{name:<42}{f1:<11.4f}{auc:<11.4f}{beats}")
for name, (f1, auc) in PAPER.items():
    print(f"{name:<42}{f1:<11.2f}{auc:<11.2f}")

fm = experiments[FINAL]["metrics"]
print(f"\nFINAL ({FINAL}): macro F1 {fm['macro_f1']:.4f} (paper best single 0.75, "
      f"{fm['macro_f1'] - 0.75:+.4f}) | macro AUC {fm['macro_roc_auc']:.4f} (paper 0.93, "
      f"{fm['macro_roc_auc'] - 0.93:+.4f})")

### 14b. Run summary — paste this output back

Saved as `summary.json` in this run's results folder on Drive.

In [ ]:
summary = {
    "run": RUN_TAG, "use_demographics": USE_DEMOGRAPHICS, "n_seeds": N_SEEDS, "n_features": int(n_feat),
    "final_model": FINAL,
    "models": {k: {"macro_f1": round(float(v["metrics"]["macro_f1"]), 4),
                   "macro_auc": round(float(v["metrics"]["macro_roc_auc"]), 4),
                   "val_macro_auc": round(float(v["val_macro_auc"]), 4)} for k, v in experiments.items()},
    "bootstrap_ci": ci_table.to_dict(orient="records"),
    "paired_final_vs_xgb": paired,
}
save_json(summary, FUSION_RESULTS / "summary.json")
print(json.dumps(summary, indent=1))
print("\nsaved:", FUSION_RESULTS / "summary.json")

## Honest notes before you report any of this

- **Feature-subset evaluation.** Fusion models are scored only on test records
  that have PTB-XL+ features (drop count printed in §7). `signal_seresnet_fulltest`
  is the full-test signal-only number — quote it next to the paper's signal-only
  row; quote fusion rows against the same subset's signal-only row.
- **Demographics** (`USE_DEMOGRAPHICS`) are an addition over the paper's
  feature set. Re-run with it `False` to report the pure Uni-G/12SL ablation.
- **Seed ensembling** (`N_SEEDS=3`) is part of the method; report the
  single-seed number too (`N_SEEDS=1`) so a reader can see how much comes
  from ensembling vs architecture.
- **No test-set selection.** Thresholds, stacking weights, fusion weights and
  the final-model choice all come from val (fold 9). Test (fold 10) is touched
  once per model.
- **Differences of < 0.01 macro AUC** between rows are within seed-to-seed
  noise on a 2k-record test set — check the bootstrap CIs in §13 before
  claiming one fusion beats another.
- The paper doesn't specify its combined architecture in detail, so the
  comparison is "our fusion vs their reported numbers", not a re-implementation.

## 15. Save the ECG models for Phase 2

Writes the bundles the Phase 2 fusion engine loads (`models/ecg/` on Drive):
the selected `features_xgb` boosters with `preprocessing="raw"`, the
imputer/scaler the joint network needs, and the signal-only thresholds. Then it
reloads each bundle and checks it reproduces the in-memory predictions.

Runs only for the main configuration (`USE_DEMOGRAPHICS = True`, `N_SEEDS = 3`),
so an ablation run never overwrites the models Phase 2 uses. Same cells as
`04b_save_ecg_feature_model.ipynb`, placed here because Colab notebooks do not
share a kernel.

In [ ]:
SAVE_FOR_PHASE2 = USE_DEMOGRAPHICS and N_SEEDS == 3
print("saving Phase 2 bundles" if SAVE_FOR_PHASE2 else
      f"skipped: run {RUN_TAG} is an ablation; Phase 2 uses the main run")

### 1. Confirm the objects are still in this kernel

In [ ]:
if SAVE_FOR_PHASE2:
    import numpy as np

    required = {
        "xgb_models": "the five one-vs-rest boosters (§10)",
        "experiments": "per-model results, including tuned thresholds (§8 helpers)",
        "FINAL": "the validation-selected model name (§13)",
        "imputer": "SimpleImputer fitted on train (§8)",
        "scaler": "StandardScaler fitted on train (§8)",
        "FEATURE_NAMES": "training column order (§7b)",
        "CLASSES": "superclass names (§3)",
        "X_val_raw": "raw val frame, for the round-trip check (§7)",
        "y_val_feat": "val labels on the feature subset (§7)",
    }
    present = set(dir())
    missing = [f"{k:14s} — {why}" for k, why in required.items() if k not in present]

    if missing:
        raise RuntimeError(
            "These objects are not in this kernel:\n  " + "\n  ".join(missing)
            + "\n\nEither run 04_ecg_feature_fusion.ipynb to the end and re-run this "
              "cell in the same session, or — if that runtime is already gone — use "
              "04c_recover_ecg_from_checkpoints.ipynb instead."
        )

    print(f"selected model (by val macro AUC): {FINAL}")
    print(f"boosters: {len(xgb_models)}   classes: {list(CLASSES)}")
    print(f"features: {len(FEATURE_NAMES)}")

### 2. Read the thresholds that `register()` already tuned

`register()` stored them under `experiments[name]["thresholds"]`, tuned on the
validation fold. They are not re-derived here — reusing the exact values keeps
this bundle consistent with the numbers notebook 04 reported.

In [ ]:
if SAVE_FOR_PHASE2:
    thr_xgb = np.asarray(experiments["features_xgb"]["thresholds"], dtype=float)
    metrics_xgb = experiments["features_xgb"]["metrics"]

    print("features_xgb thresholds (tuned on val):")
    for cls, t in zip(CLASSES, thr_xgb):
        print(f"  {cls:5s} {t:.4f}")
    print(f"\ntest macro F1  {metrics_xgb['macro_f1']:.4f}")
    print(f"test macro AUC {metrics_xgb['macro_roc_auc']:.4f}")

    if FINAL != "features_xgb":
        print(f"\n  NOTE: §13 selected '{FINAL}', not features_xgb. This notebook saves")
        print(f"  features_xgb regardless, because it is the recoverable one. If")
        print(f"  '{FINAL}' is a neural branch, 04c loads it from the .pt files.")

    # The joint model's thresholds, if that branch ran.
    thr_joint = (np.asarray(experiments["joint_gated"]["thresholds"], dtype=float)
                 if "joint_gated" in experiments else None)
    thr_signal = (np.asarray(experiments["signal_seresnet"]["thresholds"], dtype=float)
                  if "signal_seresnet" in experiments else None)
    print("\njoint_gated thresholds     :", "present" if thr_joint is not None else "absent")
    print("signal_seresnet thresholds :", "present" if thr_signal is not None else "absent")

### 3. Write the bundles to Drive

In [ ]:
if SAVE_FOR_PHASE2:
    import json, pickle, time
    from pathlib import Path

    DRIVE_ECG = Path("/content/drive/MyDrive/CardioSense/models/ecg")
    DRIVE_ECG.mkdir(parents=True, exist_ok=True)
    CLIP_SD = 5.0

    # -- the selected feature model. preprocessing="raw" is load-bearing.
    feature_bundle = {
        "model": list(xgb_models),              # five one-vs-rest binary boosters
        "preprocessing": "raw",                 # NaNs intact; do NOT impute or scale
        "imputer": None,
        "scaler": None,
        "feature_cols": list(FEATURE_NAMES),
        "classes": list(CLASSES),
        "thresholds": thr_xgb,
        "clip_sd": CLIP_SD,
        "metadata": {
            "model_version": "ecg-features_xgb-v1",
            "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "selected_model": str(FINAL),
            "selected_by": "validation macro AUC (notebook 04 §13)",
            "n_features": int(len(FEATURE_NAMES)),
            "test_macro_f1": round(float(metrics_xgb["macro_f1"]), 4),
            "test_macro_auc": round(float(metrics_xgb["macro_roc_auc"]), 4),
            "thresholds_tuned_on": "validation fold (fold 9)",
            "is_calibrated": False,
            "note": ("Fitted on the raw feature frame with NaNs intact. Imputing or "
                     "scaling before predict_proba produces plausible wrong values "
                     "and raises nothing."),
        },
    }
    with open(DRIVE_ECG / "ecg_feature_model.pkl", "wb") as handle:
        pickle.dump(feature_bundle, handle)

    # -- the preprocessor the joint network needs
    preprocessor = {
        "imputer": imputer,
        "scaler": scaler,
        "feature_cols": list(FEATURE_NAMES),
        "clip_sd": CLIP_SD,
        "thresholds": thr_joint,
        "classes": list(CLASSES),
        "metadata": {
            "model_version": "ecg-joint_gated-v1",
            "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "n_features": int(len(FEATURE_NAMES)),
            "thresholds_tuned_on": "validation fold (fold 9)",
            "is_calibrated": False,
        },
    }
    with open(DRIVE_ECG / "ecg_feature_preprocessor.pkl", "wb") as handle:
        pickle.dump(preprocessor, handle)

    written = ["ecg_feature_model.pkl", "ecg_feature_preprocessor.pkl"]

    # -- thresholds for the waveform-only adapter, which needs no preprocessor
    if thr_signal is not None:
        (DRIVE_ECG / "ecg_signal_thresholds.json").write_text(json.dumps({
            "classes": list(CLASSES),
            "thresholds": [round(float(t), 6) for t in thr_signal],
            "source": "experiments['signal_seresnet']['thresholds'] (notebook 04, val fold)",
        }, indent=2), encoding="utf-8")
        written.append("ecg_signal_thresholds.json")

    for name in written:
        path = DRIVE_ECG / name
        print(f"  {name:34s} {path.stat().st_size / 1024:>8.1f} KB")

### 4. Verify each bundle reloads and reproduces its predictions

Saving a silently wrong artifact is worse than saving nothing. Each adapter is
constructed from the files on disk — not from the objects still in memory — and
checked against the in-memory probabilities on real validation records.

In [ ]:
if SAVE_FOR_PHASE2:
    import sys

    SRC = Path.cwd() / "src"
    if str(SRC) not in sys.path:
        sys.path.insert(0, str(SRC))

    from cardiosense.fusion import ECGAdapter, resolve_ecg_adapter

    adapter = ECGAdapter(DRIVE_ECG)
    assert adapter.available, adapter.load_error
    assert adapter.preprocessing == "raw", adapter.preprocessing

    norm_idx = list(CLASSES).index("NORM")
    rows = [0, 1, 2, 3, 4]
    worst = 0.0

    for row in rows:
        raw_row = X_val_raw.iloc[row].to_dict()
        got = adapter.predict(raw_row)
        assert got.available, got.reason

        # In-memory reference: the boosters applied to the same raw row.
        matrix = X_val_raw.iloc[[row]].to_numpy(np.float32)
        reference = np.array([m.predict_proba(matrix)[0, 1] for m in xgb_models])
        want = float(1.0 - reference[norm_idx])

        worst = max(worst, abs(got.risk_score - want))
        print(f"  row {row}: adapter {got.risk_score:.6f}  in-memory {want:.6f}")

    print(f"\nlargest difference over {len(rows)} rows: {worst:.2e}")
    assert worst < 1e-6, "the reloaded bundle does not reproduce the in-memory model"
    print("bundle verified")

In [ ]:
if SAVE_FOR_PHASE2:
    # A scaled matrix must give a DIFFERENT answer -- proof the mode is load-bearing
    # rather than decorative, and that the adapter is honouring it.
    row = 0
    raw_matrix = X_val_raw.iloc[[row]].to_numpy(np.float32)
    scaled_matrix = np.clip(
        scaler.transform(imputer.transform(X_val_raw.iloc[[row]])), -CLIP_SD, CLIP_SD
    ).astype(np.float32)

    p_raw = np.array([m.predict_proba(raw_matrix)[0, 1] for m in xgb_models])
    p_scaled = np.array([m.predict_proba(scaled_matrix)[0, 1] for m in xgb_models])

    print("same boosters, same patient, two preprocessings:")
    for cls, a, b in zip(CLASSES, p_raw, p_scaled):
        print(f"  {cls:5s} raw {a:.4f}   scaled {b:.4f}   diff {abs(a - b):.4f}")
    print(f"\nmax difference {np.abs(p_raw - p_scaled).max():.4f} — and neither call errored.")
    print("That gap is the bug the `preprocessing` field exists to prevent.")

In [ ]:
if SAVE_FOR_PHASE2:
    # What the fusion engine will pick up, and under which name.
    chosen = resolve_ecg_adapter(
        DRIVE_ECG, search_roots=[Path("/content/drive/MyDrive/CardioSense")])
    print("resolved adapter :", type(chosen).__name__, f"({chosen.variant})")
    print("model_version    :", chosen.status()["model_version"])
    print("calibrated       :", chosen.is_calibrated,
          f"-> reliability capped at {chosen.reliability:.2f}")

---

### Done

| file | unlocks |
|---|---|
| `ecg_feature_model.pkl` | `features_xgb` — the model §13 selected |
| `ecg_feature_preprocessor.pkl` | the joint waveform + feature adapter |
| `ecg_signal_thresholds.json` | tuned labels for the waveform-only adapter |

**Download copies to your own machine too.** Drive is one accidental deletion
from costing a retrain.

Nothing here is calibrated: every ECG adapter reports `is_calibrated=False` and
reliability capped at 0.50. That is correct and is what the fusion layer assumes
— the ECG contribution is a ranking score, and the report says so. Running
`cardiosense.ecg.calibrate` is what lifts the cap honestly.

Next: `05_multimodal_fusion.ipynb`.